### RNN (Recurent Neural Network) : 단기메모리
- 순서가 중요, 이전사건이 다음사건에 영향을 미치는 구조
- 순서가 중요한 데이터를 --> 시계열
- 자연어, 주식, 날씨, 실시간 센서데이터
- 단점 : 장기메모리에 약함
- 개선 : LSTM(RNN에 비해서 오래 기억하지만 그래도 희석됨)
- 트랜스포머 : 집중해야할것 즉 기억해야할 중요내용만 저장(어텐션), 마스킹
### MLP
- 모든입력이 독립적, 자연어는 순서가 중요(단어의 순서)

In [1]:
from nltk.corpus import movie_reviews
reviews = [movie_reviews.raw(fileid) for fileid in movie_reviews.fileids()]
categories = [movie_reviews.categories(fileid)[0] for fileid in movie_reviews.fileids()]

In [2]:
len(reviews), len(reviews[0]), set(categories)

(2000, 4043, {'neg', 'pos'})

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
np.random.seed(42)
tf.random.set_seed(42)

In [4]:
# 사용할 단어수
max_words = 10000
tokenizer = Tokenizer(num_words=max_words, oov_token='UNK')  #빈도가 높은 10000개의 단어를 선택
tokenizer.fit_on_texts(reviews)  # 단어 인덱스 구축
x = tokenizer.texts_to_sequences(reviews)  # 인덱스를 기반으로 원래 문장을 숫자로 변환(벡터화, 개별적으로 즉 길이는 서로 다름)

In [5]:
len(x[0]), len(x[1]), x[0][:10]

(710, 240, [98, 77, 949, 4622, 131, 6, 3, 2016, 789, 3764])

In [6]:
# 학습을 위해서는 모든 문장의 길이가 같아야 함 - > 패딩  짧으면 채우고 길면 자르고
from tensorflow.keras.preprocessing.sequence import pad_sequences
maxlen = 500
x = pad_sequences(x,maxlen=maxlen,truncating='pre' )  # 앞을 자름

In [7]:
len(x[0]), len(x[1]), x[0][:10]

(500,
 500,
 array([ 911,  115,   53,   21, 5278,    5, 1387,  172,    9,  633],
       dtype=int32))

In [8]:
label_dict = {'pos' : 1, 'neg' : 0}
y = np.array([label_dict[c] for c in categories])
y[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [9]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=42,test_size=0.2)

In [10]:
x_train.shape

(1600, 500)

In [ ]:
# 일반신경망으로 분리
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten,Dense,Embedding

model = Sequential([
    Embedding(max_words,32,input_length=maxlen),    # 1600, 500 --> 1600, 500, 32
    Flatten(),
    Dense(1,activation='sigmoid')  # 확률 분포
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])
history = model.fit(x_train, y_train, epochs=10,validation_split=0.2)


c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [15]:
temp = Embedding(max_words,32,input_length=maxlen)(x_train) 
temp.shape , Flatten()(temp).shape

c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


(TensorShape([1600, 500, 32]), TensorShape([1600, 16000]))